## **Context**
As an analyst at ABC Estate Wines, we are presented with historical data encompassing the sales of different types of wines throughout the 20th century. These datasets originate from the same company but represent sales figures for distinct wine varieties. Our objective is to delve into the data, analyze trends, patterns, and factors influencing wine sales over the course of the century. By leveraging data analytics and forecasting techniques, we aim to gain actionable insights that can inform strategic decision-making and optimize sales strategies for the future

## **Objective**
The primary objective of this project is to analyze and forecast wine sales trends for the 20th century based on historical data provided by ABC Estate Wines. We aim to equip ABC Estate Wines with the necessary insights and foresight to enhance sales performance, capitalize on emerging market opportunities, and maintain a competitive edge in the wine industry..

### Import the necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import seaborn as sns
import statsmodels
import sklearn
import statsmodels.tools.eval_measures as      em
from   sklearn.metrics                 import  mean_squared_error
from   statsmodels.tsa.api             import ExponentialSmoothing, SimpleExpSmoothing, Holt
from   IPython.display                 import display

### Reading the dataset in a Time Series with proper Time frequency.

In [ ]:
df = pd.read_csv('Rose.csv',parse_dates=['YearMonth'],index_col='YearMonth')
df.head()

In [ ]:
df.shape

In [ ]:
# copying data to another variable to avoid any changes to original data
data = df.copy()

### Checking the data types of the columns for the dataset.

In [ ]:
data.info()

### Checking for missing values

In [ ]:
# checking for null values
data.isnull().sum()

### Imputing Missing values using Interpolation

In [ ]:
# Filter only 1994 data
data_1994 = data[data.index.year == 1994]
data_1994

In [ ]:
# Apply spline interpolation to the Rose column
data_1994["Rose"] = data_1994["Rose"].interpolate(method="spline", order=1)

In [ ]:
# Merge back with original dataset
data.update(data_1994)

In [ ]:
data.isnull().sum()

All the Missing values are dealt.

### Checking for Duplicates

In [ ]:
data.duplicated().sum()

### These are not duplicates as these are Sales values. 

In [ ]:
data.describe(include="all").T

### Ploting the Time Series Data.

In [ ]:
from pylab import rcParams

In [ ]:
rcParams['figure.figsize'] = 15,8

In [ ]:
data.plot();
plt.grid()

* Sales follow repeated cycles, indicating possible seasonal patterns—likely driven by consumer demand variations across months.
* Some periods show sharp declines, suggesting market disruptions, supply chain variations, or external influences.

### Plot a boxplot to understand the variation of Rose wine Sales to months across years.

In [ ]:
sns.boxplot(x=data.index.month,y=data['Rose'])
plt.grid();

* Month 12 (December) shows the highest median sales, suggesting peak demand during the holiday season.
* Months 1-6 have relatively lower sales, possibly indicating slower consumption post-holiday.

### Plot a boxplot to understand the variation of Rose wine Sales to Years across years

In [ ]:
sns.boxplot(x=data.index.year,y=data['Rose'])
plt.grid();

* The median sales figures appear to drop from 1980 to 1995, indicating an overall decline in demand.

### Plot a graph of monthly Rose wine Sales across years.

In [ ]:
monthly_Rose_across_years = pd.pivot_table(data, values = 'Rose', columns = data.index.month_name(), index = data.index.year)
monthly_Rose_across_years

* **Peak Sales in December**
* December consistently has higher sales figures, likely due to holiday demand.
* Certain months like August, November, and December show elevated sales, reinforcing seasonal demand patterns.

In [ ]:
monthly_Rose_across_years.plot()
plt.grid()
plt.legend(loc='best');

**Seasonal Sales Variations**

* The December line is notably higher, suggesting peak demand during holiday seasons.
* February and January show lower sales, suggesting post-holiday demand reduction.

**Declining Long-Term Trend**

* Most monthly sales trends show a downward movement between 1980-1994.
* Possible factors: changing consumer preferences, market competition, or supply-chain adjustments.

### Plot a time series monthplot to understand the spread of Rose Wine Sales across different years and within different months across years.

In [ ]:
from statsmodels.graphics.tsaplots import month_plot

month_plot(data['Rose'],ylabel='Rose Wine Sales')
plt.grid();

**Fluctuating Sales Patterns**

* Sales rise and fall cyclically, suggesting seasonal demand.

* Peaks are visible in certain months, possibly driven by events or holidays.

**Notable Spikes**

* Specific months show sharp increases, indicating higher demand in those periods.

**Gradual Growth Over Time**

* While fluctuations exist, sales appear to decrease gradually, showing possible long-term market collapse.

### Plot the Empirical Cumulative Distribution of Sprakling Wine Sales.

In [ ]:
# statistics
from statsmodels.distributions.empirical_distribution import ECDF

plt.figure(figsize = (18, 8))
cdf = ECDF(data['Rose'])
plt.plot(cdf.x, cdf.y, label = "statmodels");
plt.grid()
plt.xlabel('Rose Wine Sales');
plt.ylabel('pct_of_sales');

* Sales are concentrated around lower and mid-range values, indicating that most purchases occur within typical demand levels.
* As the graph approaches higher sales numbers (200+ units), the cumulative percentage nears 100%, meaning nearly all sales are accounted for in this range.

### Plot a graph of the average and percentage change of Sales of Rose wine across years

In [ ]:
# group by date and get average Customers, and precent change
average_Rose_sales    = data.groupby(data.index)["Rose"].mean()
pct_change_in_sales = data.groupby(data.index)["Rose"].sum().pct_change()

fig, (axis1,axis2) = plt.subplots(2,1,sharex=True,figsize=(15,8))

# plot average Sales over time(year-month)
ax1 = average_Rose_sales.plot(legend=True,ax=axis1,marker='o',title="average_Rose_sales")

ax1.set_xticks(range(len(average_Rose_sales)))
ax1.set_xticklabels(average_Rose_sales.index.tolist(), rotation=90)

# plot precent change for Sales over time(year-month)
ax2 = pct_change_in_sales.plot(legend=True,ax=axis2,marker='o',rot=90,colormap="summer",title="pct_change_in_sales")

* Sales peaked around 1981, surpassing 250 units, but show a gradual decline leading to 1995.
* The overall trend suggests declining demand, possibly influenced by market preferences, competition, or economic shifts
* Highly volatile changes, with fluctuations ranging from -0.6 to 0.8.
* Periods of strong positive growth (1983) indicate temporary surges in sales, while sharp declines (1990s) suggest downturns in demand.

### Quarterly Plot

In [ ]:
data_quarterly_sum = data.resample('Q').sum()
data_quarterly_sum.head()

In [ ]:
data_quarterly_sum.plot();
plt.grid()

In [ ]:
data_quarterly_mean= data.resample('Q').mean()
data_quarterly_mean.head()

In [ ]:
data_quarterly_mean.plot();
plt.grid()

In [ ]:
data.plot()
plt.grid();

## Decomposing the Time Series

### Additive Model

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
decomposition = seasonal_decompose(data,model='additive')
decomposition.plot();

* From the plot we can see that alot of residuals are situated around 0 

### Let's check for Multiplicative Decompostion

In [ ]:
decomposition = seasonal_decompose(data['Rose'],model='multiplicative')
decomposition.plot();

* For the multiplicative series, we see that a lot of residuals are located around 1. Thus, Multiplicative Decomposition is the right way to decompose the time series

In [ ]:
trend = decomposition.trend
seasonality = decomposition.seasonal
residual = decomposition.resid

In [ ]:
print('Trend','\n',trend.round(2).head(12),'\n')
print('Seasonality','\n',seasonality.round(2).head(12),'\n')
print('Residual','\n',residual.round(2).head(12),'\n')

In [ ]:
deaseasonalized_ts = trend + residual
deaseasonalized_ts.round(2).head(12)

In [ ]:
data.plot()
deaseasonalized_ts.plot()
plt.legend(["Original Time Series", "Time Series without Seasonality Component"]);

* Without seasonality, sales appear more stable over time.

* There is downward trend, suggesting consistent decrese in demand.

### Checking for stationarity of the whole Time Series data.

In [ ]:
## Test for stationarity of the series - Dicky Fuller test

from statsmodels.tsa.stattools import adfuller
def test_stationarity(timeseries):
    
    #Determing rolling statistics
    rolmean = timeseries.rolling(window=7).mean() #determining the rolling mean
    rolstd = timeseries.rolling(window=7).std()   #determining the rolling standard deviation

    #Plot rolling statistics:
    orig = plt.plot(timeseries, color='blue',label='Original')
    mean = plt.plot(rolmean, color='red', label='Rolling Mean')
    std = plt.plot(rolstd, color='black', label = 'Rolling Std')
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.show(block=False)
    
    #Perform Dickey-Fuller test:
    print ('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print (dfoutput,'\n')

In [ ]:
test_stationarity(data['Rose'])

* We see that at p-value is greater than alpha value which is 0.05 or 5% significant level, This tells this Time Series is non-stationary.
* Conclusion: The dataset is not stationary, meaning sales exhibit long-term patterns or trends.

### Let us take a difference of order 1 and check whether the Time Series is stationary or not.

In [ ]:
test_stationarity(data['Rose'].diff().dropna())

* The Rolling Mean & Standard Deviation plot illustrates the stabilized trend of Rose wine sales after applying first-order differencing, making the series stationary.
* ### We see that at $\alpha$ = 0.05 the Time Series is indeed stationary.

### Ploting the Autocorrelation function plots

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
plot_acf(data['Rose'],lags=50)
plot_acf(data['Rose'].diff().dropna(),lags=50,title='Differenced Data Autocorrelation')
plt.show()

* From the above plots, we can confirms the seasonality in the data.

## Spliting the data into train and test and plot the training and test data.

### Training Data is till the end of 1990. Test Data is from the beginning of 1991 to the last time stamp provided.

In [ ]:
train=data[data.index.year <= 1991]
test=data[data.index.year > 1991]

In [ ]:
print(train.shape)
print(test.shape)

In [ ]:
print(train.head(),'\n')
print(train.tail(),'\n\n')
print(test.head(),'\n')
print(test.tail(),'\n')

# Building different models and comparing the accuracy metrics.

## Model 1: Linear Regression

In [ ]:
# Create numerical time indices
train_time = list(range(1, len(train) + 1))  # Sequential index for training
test_time = list(range(len(train) + 1, len(train) + len(test) + 1))  # Continuing index for test

# Print results
print("Training Time Instances:\n", train_time)
print("Test Time Instances:\n", test_time)

In [ ]:
LinearRegression_train = train.copy()
LinearRegression_test = test.copy()

In [ ]:
LinearRegression_train['time'] = train_time
LinearRegression_test['time'] = test_time

print('First few rows of Training Data')
display(LinearRegression_train.head())
print('Last few rows of Training Data')
display(LinearRegression_train.tail())
print('First few rows of Test Data')
display(LinearRegression_test.head())
print('Last few rows of Test Data')
display(LinearRegression_test.tail())

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

lr.fit(LinearRegression_train[['time']],LinearRegression_train['Rose'])

In [ ]:
train_predictions_model1         = lr.predict(LinearRegression_train[['time']])
LinearRegression_train['RegOnTime'] = train_predictions_model1

test_predictions_model1         = lr.predict(LinearRegression_test[['time']])
LinearRegression_test['RegOnTime'] = test_predictions_model1

plt.plot( train['Rose'], label='Train')
plt.plot(test['Rose'], label='Test')
plt.plot(LinearRegression_test['RegOnTime'], label='Regression On Time_Test Data')

plt.legend(loc='best')
plt.grid();

#### Defining the functions for calculating the accuracy metrics.

In [ ]:
from sklearn import metrics

### Model Evaluation

In [ ]:
rmse_model1_test = metrics.mean_squared_error(test['Rose'],test_predictions_model1,squared=False)
print("For RegressionOnTime forecast on the Test Data,  RMSE is %3.3f " %(rmse_model1_test))

In [ ]:
resultsDf = pd.DataFrame({'RMSE': [rmse_model1_test]},index=['RegressionOnTime'])
resultsDf

## Method 2: Simple Average

In [ ]:
SimpleAverage_train = train.copy()
SimpleAverage_test = test.copy()

In [ ]:
SimpleAverage_test['mean_forecast'] = train['Rose'].mean()
SimpleAverage_test.head()

In [ ]:
plt.plot(SimpleAverage_train['Rose'], label='Train')
plt.plot(SimpleAverage_test['Rose'], label='Test')

plt.plot(SimpleAverage_test['mean_forecast'], label='Simple Average on Test Data')

plt.legend(loc='best')
plt.title("Simple Average Forecast")
plt.grid();

### Model Evaluation

In [ ]:
rmse_model2_test = metrics.mean_squared_error(test['Rose'],SimpleAverage_test['mean_forecast'],squared=False)
print("For Simple Average forecast on the Test Data,  RMSE is %3.3f" %(rmse_model2_test))

In [ ]:
resultsDf_2 = pd.DataFrame({'RMSE': [rmse_model2_test]}
                           ,index=['SimpleAverageModel'])

resultsDf = pd.concat([resultsDf, resultsDf_2])
resultsDf

## Method 3: Moving Average(MA)

###### For Moving Average, we are going to average over the entire data.

In [ ]:
MovingAverage = data.copy()
MovingAverage.head()

#### Trailing moving averages

In [ ]:

MovingAverage['Trailing_2'] = MovingAverage['Rose'].rolling(2).mean()
MovingAverage['Trailing_4'] = MovingAverage['Rose'].rolling(4).mean()
MovingAverage['Trailing_6'] = MovingAverage['Rose'].rolling(6).mean()
MovingAverage['Trailing_9'] = MovingAverage['Rose'].rolling(9).mean()

MovingAverage.head()

In [ ]:
## Plotting on the whole data

plt.plot(MovingAverage['Rose'], label='Train')
plt.plot(MovingAverage['Trailing_2'], label='2 Point Moving Average')
plt.plot(MovingAverage['Trailing_4'], label='4 Point Moving Average')
plt.plot(MovingAverage['Trailing_6'],label = '6 Point Moving Average')
plt.plot(MovingAverage['Trailing_9'],label = '9 Point Moving Average')

plt.legend(loc = 'best')
plt.grid();

Let us split the data into train and test and plot this Time Series. The window of the moving average is need to be carefully selected as too big a window will result in not having any test set as the whole series might get averaged over.

In [ ]:
#Creating train and test set 
trailing_MovingAverage_train=MovingAverage[0:int(len(MovingAverage)*0.7)] 
trailing_MovingAverage_test=MovingAverage[int(len(MovingAverage)*0.7):]

In [ ]:
print(f"Training set size: {len(trailing_MovingAverage_train)}")
print(f"Test set size: {len(trailing_MovingAverage_test)}")

In [ ]:
print(trailing_MovingAverage_train.isnull().sum())
print(trailing_MovingAverage_test.isnull().sum())

In [ ]:
trailing_MovingAverage_test = trailing_MovingAverage_test.iloc[:len(test)]


In [ ]:
## Plotting on both the Training and Test data

plt.figure(figsize=(16,8))
plt.plot(trailing_MovingAverage_train['Rose'], label='Train')
plt.plot(trailing_MovingAverage_test['Rose'], label='Test')


plt.plot(trailing_MovingAverage_test['Trailing_2'], label='2 Point Trailing Moving Average on Test Set')
plt.plot(trailing_MovingAverage_test['Trailing_4'], label='4 Point Trailing Moving Average on Test Set')
plt.plot(trailing_MovingAverage_test['Trailing_6'],label = '6 Point Trailing Moving Average on Test Set')
plt.plot(trailing_MovingAverage_test['Trailing_9'],label = '9 Point Trailing Moving Average on Test Set')

plt.legend(loc = 'best')
plt.grid();

### Model Evaluation

##### Done only on the test data.

In [ ]:
## Test Data - RMSE  --> 2 point Trailing MA

rmse_model4_test_2 = metrics.mean_squared_error(test['Rose'],trailing_MovingAverage_test['Trailing_2'],squared=False)
print("For 2 point Moving Average Model forecast on the Training Data,  RMSE is %3.3f" %(rmse_model4_test_2))

## Test Data - RMSE --> 4 point Trailing MA

rmse_model4_test_4 = metrics.mean_squared_error(test['Rose'],trailing_MovingAverage_test['Trailing_4'],squared=False)
print("For 4 point Moving Average Model forecast on the Training Data,  RMSE is %3.3f" %(rmse_model4_test_4))

## Test Data - RMSE --> 6 point Trailing MA

rmse_model4_test_6 = metrics.mean_squared_error(test['Rose'],trailing_MovingAverage_test['Trailing_6'],squared=False)
print("For 6 point Moving Average Model forecast on the Training Data,  RMSE is %3.3f" %(rmse_model4_test_6))

## Test Data - RMSE --> 9 point Trailing MA

rmse_model4_test_9 = metrics.mean_squared_error(test['Rose'],trailing_MovingAverage_test['Trailing_9'],squared=False)
print("For 9 point Moving Average Model forecast on the Training Data,  RMSE is %3.3f" %(rmse_model4_test_9))

In [ ]:
resultsDf_3 = pd.DataFrame({'RMSE': [rmse_model4_test_2,rmse_model4_test_4
                                          ,rmse_model4_test_6,rmse_model4_test_9]}
                           ,index=['2pointTrailingMovingAverage','4pointTrailingMovingAverage'
                                   ,'6pointTrailingMovingAverage','9pointTrailingMovingAverage'])

resultsDf = pd.concat([resultsDf, resultsDf_3])
resultsDf

In [ ]:
## Plotting on both Training and Test data

plt.plot(train['Rose'], label='Train')
plt.plot(test['Rose'], label='Test')

plt.plot(LinearRegression_test['RegOnTime'], label='Regression On Time_Training Data')

plt.plot(SimpleAverage_test['mean_forecast'], label='Simple Average on Test Data')

plt.plot(trailing_MovingAverage_test['Trailing_2'], label='2 Point Trailing Moving Average on Training Set')


plt.legend(loc='best')
plt.title("Model Comparison Plots")
plt.grid();

## Method 4.1: Simple Exponential Smoothing

In [ ]:
# create class
model_SES = SimpleExpSmoothing(train,initialization_method='estimated')

In [ ]:
# Fitting the Simple Exponential Smoothing model and asking python to choose the optimal parameters
model_SES_autofit = model_SES.fit(optimized=True)

In [ ]:
## Let us check the parameters

model_SES_autofit.params

In [ ]:
# Using the fitted model on the training set to forecast on the test set
SES_predict = model_SES_autofit.forecast(steps=len(test))
SES_predict

In [ ]:
## Plotting the Training data, Test data and the forecasted values

plt.plot(train, label='Train')
plt.plot(test, label='Test')

plt.plot(SES_predict, label='Alpha =0.0987 Simple Exponential Smoothing predictions on Test Set')

plt.legend(loc='best')
plt.grid()
plt.title('Alpha = 0.0987 Predictions');

In [ ]:
## Mean Absolute Percentage Error (MAPE) - Function Definition

def MAPE(y_true, y_pred):
    return np.mean((np.abs(y_true-y_pred))/(y_true))*100

In [ ]:
print('SES RMSE:',mean_squared_error(test.values,SES_predict.values,squared=False))
#different way to calculate RMSE
print('SES RMSE (calculated using statsmodels):',em.rmse(test.values,SES_predict.values)[0])

In [ ]:
resultsDf_4 = pd.DataFrame({'RMSE': [em.rmse(test.values,SES_predict.values)[0]]},index=['Simple Exponential Smoothing: Alpha =0.0987'])

resultsDf = pd.concat([resultsDf, resultsDf_4])
resultsDf

## Method 4.2: Double Exponential Smoothing

In [ ]:
# Initializing the Double Exponential Smoothing Model
model_DES = Holt(train,initialization_method='estimated')
# Fitting the model
model_DES = model_DES.fit()

print('')
print('==Holt model Exponential Smoothing Estimated Parameters ==')
print('')
print(model_DES.params)

In [ ]:
# Forecasting using this model for the duration of the test set
DES_predict =  model_DES.forecast(len(test))
DES_predict

In [ ]:
## Plotting the Training data, Test data and the forecasted values

plt.plot(train, label='Train')
plt.plot(test, label='Test')

plt.plot(SES_predict, label='Alpha =0.0987 Simple Exponential Smoothing predictions on Test Set')
plt.plot(DES_predict, label='Alpha=1.490,Beta=1.661:Double Exponential Smoothing predictions on Test Set')

plt.legend(loc='best')
plt.grid()
plt.title('Simple and Double Exponential Smoothing Predictions');

* We see that the double exponential smoothing is picking up the trend component along with the level component as well.

In [ ]:
print('DES RMSE:',mean_squared_error(test.values,DES_predict.values,squared=False))

In [ ]:
resultsDf_4_2 = pd.DataFrame({'RMSE': [mean_squared_error(test.values,DES_predict.values,squared=False)]}
                           ,index=['Double Exponential Smoothing:Alpha=1.490,Beta=1.661'])

resultsDf = pd.concat([resultsDf, resultsDf_4_2])
resultsDf

## Method 4.3: Holt-Winters - ETS(A, A, A) - Holt Winter's linear method with Multiplicative errors

In [ ]:
# Initializing the Double Exponential Smoothing Model
model_TES = ExponentialSmoothing(train,trend='multiplicative',seasonal='multiplicative',initialization_method='estimated')
# Fitting the model
model_TES = model_TES.fit()

print('')
print('==Holt Winters model Exponential Smoothing Estimated Parameters ==')
print('')
print(model_TES.params)

In [ ]:
# Forecasting using this model for the duration of the test set
TES_predict =  model_TES.forecast(len(test))
TES_predict

In [ ]:
## Plotting the Training data, Test data and the forecasted values

plt.plot(train, label='Train')
plt.plot(test, label='Test')

plt.plot(SES_predict, label='Alpha =0.0987 Simple Exponential Smoothing predictions on Test Set')
plt.plot(DES_predict, label='Alpha=1.490,Beta=1.661:Double Exponential Smoothing predictions on Test Set')
plt.plot(TES_predict, label='Alpha=0.055,Beta=0.031,Gamma=0.0003:Triple Exponential Smoothing predictions on Test Set')

plt.legend(loc='best')
plt.grid()
plt.title('Simple,Double and Triple Exponential Smoothing Predictions');

* We see that the Triple Exponential Smoothing is picking up the seasonal component as well.

In [ ]:
print('TES RMSE:',mean_squared_error(test.values,TES_predict.values,squared=False))

In [ ]:
resultsDf_temp = pd.DataFrame({'RMSE': [mean_squared_error(test.values,TES_predict.values,squared=False)]}
                           ,index=['Triple Exponential Smoothing: Alpha=0.055,Beta=0.031,Gamma=0.0003'])

resultsDf = pd.concat([resultsDf, resultsDf_temp])
resultsDf

## Method 5: ARMA Model

### Testing the training data for stationarity using the Augmented Dickey-Fuller (ADF) test at $\alpha$ = 0.05.
### Training Data stationarity for Rose Wine Sales

In [ ]:
## Test for stationarity of the series - Dicky Fuller test

from statsmodels.tsa.stattools import adfuller
def test_stationarity(timeseries):
    
    #Determing rolling statistics
    rolmean = timeseries.rolling(window=7).mean()
    rolstd = timeseries.rolling(window=7).std()

    #Plot rolling statistics:
    orig = plt.plot(timeseries, color='blue',label='Original')
    mean = plt.plot(rolmean, color='red', label='Rolling Mean')
    std = plt.plot(rolstd, color='black', label = 'Rolling Std')
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.show(block=False)
    
    #Perform Dickey-Fuller test:
    print ('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print (dfoutput,'\n')

In [ ]:
test_stationarity(train['Rose'])

### We see that the series is not stationary at $\alpha$ = 0.05.

In [ ]:
test_stationarity(train['Rose'].diff().dropna())

* We see that after taking a difference of order 1 the series have become stationary at $\alpha$ = 0.05.

## Building an Automated version of an <font color='blue'>ARMA model</font> for which the best parameters are selected in accordance with the lowest Akaike Information Criteria (AIC).

In [ ]:
import itertools
p = q = range(0, 5)
d= range(1)
pdq = list(itertools.product(p, d, q))
print('Some parameter combinations for the Model...')
for i in range(1,len(pdq)):
    print('Model: {}'.format(pdq[i]))

In [ ]:
# Creating an empty Dataframe with column names only
ARMA_AIC= pd.DataFrame(columns=['param', 'AIC'])
ARMA_AIC

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

for param in pdq:
    ARMA_model = ARIMA(train['Rose'].values,order=param).fit()
    print('ARIMA{} - AIC:{}'.format(param,ARMA_model.aic))
    
    new_row = pd.DataFrame({'param': [param], 'AIC': [ARMA_model.aic]})
    ARMA_AIC = pd.concat([ARMA_AIC, new_row], ignore_index=True)

In [ ]:
## Sort the above AIC values in the ascending order to get the parameters for the minimum AIC value

ARMA_AIC.sort_values(by='AIC',ascending=True).head()

In [ ]:
auto_ARIMA = ARIMA(train['Rose'], order=(3,0,3))

results_auto_ARIMA = auto_ARIMA.fit()

print(results_auto_ARIMA.summary())

### Predicting on the Test of Rose wine Set using this model and evaluate the model.

In [ ]:
predicted_auto_ARIMA = results_auto_ARIMA.forecast(steps=len(test))

In [ ]:
from sklearn.metrics import  mean_squared_error
rmse = mean_squared_error(test['Rose'],predicted_auto_ARIMA ,squared=False)
print(rmse)

In [ ]:
resultsDf_temp = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['ARMA(3,0,3)'])

resultsDf = pd.concat([resultsDf, resultsDf_temp])
resultsDf

## Method 6: ARIMA Model

## Building an Automated version of an <font color='blue'>ARIMA model</font> for which the best parameters are selected in accordance with the lowest Akaike Information Criteria (AIC).

In [ ]:
import itertools
p = q = range(0, 5)
d= range(1,3)
pdq = list(itertools.product(p, d, q))
print('Some parameter combinations for the Model...')
for i in range(1,len(pdq)):
    print('Model: {}'.format(pdq[i]))

In [ ]:
# Creating an empty Dataframe with column names only
ARIMA_AIC = pd.DataFrame(columns=['param', 'AIC'])
ARIMA_AIC

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

for param in pdq:
    ARIMA_model = ARIMA(train['Rose'].values,order=param).fit()
    print('ARIMA{} - AIC:{}'.format(param,ARIMA_model.aic))
    
    new_row = pd.DataFrame({'param': [param], 'AIC': [ARIMA_model.aic]})
    ARIMA_AIC = pd.concat([ARIMA_AIC, new_row], ignore_index=True)

In [ ]:
## Sort the above AIC values in the ascending order to get the parameters for the minimum AIC value

ARIMA_AIC.sort_values(by='AIC',ascending=True).head()

In [ ]:
auto_ARIMA = ARIMA(train['Rose'], order=(4,2,4))

results_auto_ARIMA = auto_ARIMA.fit()

print(results_auto_ARIMA.summary())

### Predicting on the Test Set using this model and evaluate the model.

In [ ]:
predicted_auto_ARIMA = results_auto_ARIMA.forecast(steps=len(test))

In [ ]:
from sklearn.metrics import  mean_squared_error
rmse = mean_squared_error(test['Rose'],predicted_auto_ARIMA,squared=False)
print(rmse)

In [ ]:
temp_resultsDf0 = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['ARIMA(4,2,4)'])


resultsDf = pd.concat([resultsDf,temp_resultsDf0])
resultsDf

## Method 7: SARIMA Model

### Building an Automated version of a <font color='blue'>SARIMA</font> model for which the best parameters are selected in accordance with the lowest Akaike Information Criteria (AIC).

### SARIMA Model for Rose Wine Sales 

In [ ]:
plot_acf(data['Rose'].diff().dropna(),lags=50,title='Differenced Data Autocorrelation')
plt.show()

#### Eventhough data says Seasonality is 12 month, just checking for 6-Month Seasonality

In [ ]:
import itertools
p = q = range(0, 3)
d= range(1,2)
D = range(0,1)
pdq = list(itertools.product(p, d, q))
model_pdq = [(x[0], x[1], x[2], 6) for x in list(itertools.product(p, D, q))]
print('Examples of some parameter combinations for Model...')
for i in range(1,len(pdq)):
    print('Model: {}{}'.format(pdq[i], model_pdq[i]))

In [ ]:
SARIMA_AIC = pd.DataFrame(columns=['param','seasonal', 'AIC'])
SARIMA_AIC

In [ ]:
import statsmodels.api as sm

for param in pdq:
    for param_seasonal in model_pdq:
        SARIMA_model = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
            
        results_SARIMA = SARIMA_model.fit(maxiter=1000)
        print('SARIMA{}x{} - AIC:{}'.format(param, param_seasonal, results_SARIMA.aic))

        new_row = pd.DataFrame({'param':[param],'seasonal':[param_seasonal] ,'AIC': results_SARIMA.aic})
        SARIMA_AIC = pd.concat([SARIMA_AIC, new_row], ignore_index=True)

In [ ]:
SARIMA_AIC.sort_values(by=['AIC']).head()

In [ ]:
import statsmodels.api as sm

auto_SARIMA_6 = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                order=(1, 1, 2),
                                seasonal_order=(2, 0, 2, 6),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
results_auto_SARIMA_6 = auto_SARIMA_6.fit(maxiter=1000)
print(results_auto_SARIMA_6.summary())

In [ ]:
results_auto_SARIMA_6.plot_diagnostics()
plt.show()

**Standardized Residuals**

* Residuals fluctuate around zero, showing the model captures the overall trend well.

* Some mild systematic variations, indicating potential room for fine-tuning.

**Histogram & Density Plot**

* Residuals are approximately normally distributed, but with slight skew.

* The Kernel Density Estimate (KDE) curve is slightly off from a perfect normal distribution, suggesting mild deviations.

**Q-Q Plot**

* Residuals generally align with normal quantiles, but deviations in the tails suggest minor non-normality.

**Correlogram**

* Some autocorrelation is present in residuals, meaning certain patterns were not fully modeled.

* A more refined seasonal structure (perhaps 12-month seasonality) might improve results.

### Predict on the Test Set using this model and evaluate the model.

In [ ]:
predicted_auto_SARIMA_6 = results_auto_SARIMA_6.get_forecast(steps=len(test))

In [ ]:
predicted_auto_SARIMA_6.summary_frame(alpha=0.05).head()

In [ ]:
rmse = mean_squared_error(test['Rose'],predicted_auto_SARIMA_6.predicted_mean,squared=False)
print(rmse)

In [ ]:
temp_resultsDf = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['SARIMA(1, 1, 2)(2, 0, 2, 6) 6-Month Seasonality'])


resultsDf = pd.concat([resultsDf,temp_resultsDf])

resultsDf

### SARIMA Model with 12-Month Seasonality

In [ ]:
import itertools
p = q = range(0, 3)
d= range(1,2)
D = range(0,1)
pdq = list(itertools.product(p, d, q))
model_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, D, q))]
print('Examples of some parameter combinations for Model...')
for i in range(1,len(pdq)):
    print('Model: {}{}'.format(pdq[i], model_pdq[i]))

In [ ]:
SARIMA_AIC = pd.DataFrame(columns=['param','seasonal', 'AIC'])
SARIMA_AIC

In [ ]:
import statsmodels.api as sm

for param in pdq:
    for param_seasonal in model_pdq:
        SARIMA_model = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
            
        results_SARIMA = SARIMA_model.fit(maxiter=1000)
        print('SARIMA{}x{} - AIC:{}'.format(param, param_seasonal, results_SARIMA.aic))

        new_row = pd.DataFrame({'param':[param],'seasonal':[param_seasonal] ,'AIC': results_SARIMA.aic})
        SARIMA_AIC = pd.concat([SARIMA_AIC, new_row], ignore_index=True)

In [ ]:
SARIMA_AIC.sort_values(by=['AIC']).head()

In [ ]:
import statsmodels.api as sm

auto_SARIMA_12 = sm.tsa.statespace.SARIMAX(train['Rose'].values,
                                order=(0, 1, 2),
                                seasonal_order=(2, 0, 2, 12),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
results_auto_SARIMA_12 = auto_SARIMA_12.fit(maxiter=1000)
print(results_auto_SARIMA_12.summary())

In [ ]:
results_auto_SARIMA_12.plot_diagnostics()
plt.show()

**Standardized Residuals**

* Residuals fluctuate around zero, indicating a reasonable model fit.

* No severe trends, suggesting no major bias in predictions.

**Histogram & Density Plot**

* Residuals roughly follow a normal distribution, aligning with model assumptions.

* The Kernel Density Estimate (KDE) curve closely resembles a normal distribution.

**Q-Q Plot**

* Residuals mostly align with the normal distribution (diagonal line).

* Some deviations in the tails suggest mild non-normality.

**Correlogram**

* Residual autocorrelation is insignificant, confirming no strong patterns are left in the data.

* This implies the SARIMA model captures dependencies well.

### From the model diagnostics plot, we can see that all the individual diagnostics plots almost follow the theoretical numbers and thus we cannot develop any pattern from these plots. 

### Predict on the Test Set using this model and evaluate the model.

In [ ]:
predicted_auto_SARIMA_12 = results_auto_SARIMA_12.get_forecast(steps=len(test))

In [ ]:
predicted_auto_SARIMA_12.summary_frame(alpha=0.05).head()

In [ ]:
rmse = mean_squared_error(test['Rose'],predicted_auto_SARIMA_12.predicted_mean,squared=False)
print(rmse)

In [ ]:
temp_resultsDf = pd.DataFrame({'RMSE': [rmse]}
                           ,index=['SARIMA(1, 1, 2)(1,0,2,12) 12-Month Seasonality'])


resultsDf = pd.concat([resultsDf,temp_resultsDf])

resultsDf

In [ ]:
resultsDf.sort_values(by=['RMSE'])

**Exponential Smoothing Models**

* Triple Exponential Smoothing (RMSE: 11.24): The best model, effectively capturing trend, level, and seasonality.

* Double Exponential Smoothing (RMSE: 14.91): Performs well but lacks seasonal adjustments, leading to slightly higher RMSE.

**Moving Average Models**

* 6-Point Moving Average (RMSE: 20.45) performs better than 9-Point (RMSE: 20.57) and 4-Point (RMSE: 20.69).

* Trailing Moving Averages smooth data, but their higher RMSE suggests they don’t capture seasonality well enough.

**SARIMA & ARIMA Models**

* SARIMA(12) (RMSE: 23.10) outperforms SARIMA(6) (RMSE: 23.71): A 12-month seasonality captures long-term trends better.

* ARIMA(2,1,3) (RMSE: 29.72): Differencing impacts accuracy, making ARIMA less reliable than SARIMA.


## Building the most optimum model on the Rose Wine Full Data.

## Final Model using TES

In [ ]:
# Define date range for forecasting
forecast_index = pd.date_range(start="1991-08-01", end="2005-12-01", freq="M")

In [ ]:
# Forecasting using this model for the duration of the test set
TES_predict =  model_TES.forecast(steps=len(forecast_index))
TES_predict

In [ ]:
# Plot observed data
axis = data['Rose'].plot(label='Observed')

# Plot TES forecast
TES_predict.plot(ax=axis, label='TES Forecast', alpha=0.8)

# Compute confidence intervals (Assuming ±5% variation for visualization)
ci_lower = TES_predict * 0.95
ci_upper = TES_predict * 1.05

# Plot confidence bands
axis.fill_between(TES_predict.index, ci_lower, ci_upper,color='k', alpha=0.1)

# Labels and legend
axis.set_xlabel('YearMonth')
axis.set_ylabel('Rose Wine Sales')
plt.legend(loc='best')
plt.title("Triple Exponential Smoothing Forecast with Confidence Band")
plt.grid()
plt.show()

**Long-Term Demand Outlook**

* The forecasted trajectory suggests declining consumer interest in Rosé wine beyond 1995.
* While seasonal peaks remain visible, the overall trend weakens, hinting at potential market saturation or shifting consumer preferences.


## Final Model using SARIMA(12)

In [ ]:
full_data_model = sm.tsa.statespace.SARIMAX(data['Rose'],
                                order=(1,1,2),
                                seasonal_order=(1, 0, 2, 12),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
results_full_data_model = full_data_model.fit(maxiter=1000)
print(results_full_data_model.summary())

In [ ]:
results_full_data_model.plot_diagnostics();

## Evaluating 12-month Rose Wine model on the whole and predict Entire 20th Century i.e., unitil year 2005 into the future

In [ ]:
predicted_manual_SARIMA_12_full_data = results_full_data_model.get_forecast(steps=172)

In [ ]:
predicted_manual_SARIMA_12_full_data.summary_frame(alpha=0.05).head()

In [ ]:
rmse = mean_squared_error(data['Rose'],results_full_data_model.fittedvalues,squared=False)
print('RMSE of the Full Model',rmse)

In [ ]:
pred_full_manual_SARIMA_date = predicted_manual_SARIMA_12_full_data.summary_frame(alpha=0.05).set_index(pd.date_range(start='1991-08-01',end='2005-12-01', freq='M'))

In [ ]:
# plot the forecast along with the confidence band

axis = data['Rose'].plot(label='Observed')
pred_full_manual_SARIMA_date['mean'].plot(ax=axis, label='Forecast', alpha=0.8)
axis.fill_between(pred_full_manual_SARIMA_date.index, pred_full_manual_SARIMA_date['mean_ci_lower'], 
                  pred_full_manual_SARIMA_date['mean_ci_upper'], color='k', alpha=0.1)
axis.set_xlabel('YearMonth')
axis.set_ylabel('Rose')
plt.title("SARIMA Forecast with Confidence Band")
plt.legend(loc='best')
plt.show()

**Continued Seasonal Trends**

* The model maintains periodic peaks and troughs, reflecting annual sales cycles.

* Suggests demand reducing patterns into the future.

**Confidence Intervals & Forecast Reliability**

* The shaded region around the forecast indicates uncertainty bounds, ensuring realistic expectations.

* Wider intervals suggest greater uncertainty in later years, a natural effect in time series forecasting.

**Long-Term Sales Stability**

* Early years (1980 - 1990) show higher variability, with fluctuations in demand.
* Later years (1995 - 2005) indicate a gradual decline, suggesting a shift in consumer preferences or external market changes. If this trend persists, stability in sales may weaken, requiring adjustments in production and marketing efforts.

# **Comparing TES vs. SARIMA**

* TES has lower RMSE, confirming its better predictive accuracy than SARIMA(12).
* TES captures trend, level, and seasonality, while SARIMA relies mainly on seasonal autoregression patterns.
* TES adapts better to subtle sales shifts, making it ideal for longer-term forecasting.

# **Actionable Business Insights & Strategic Recommendations for ABC Estate Wines**

**Declining Long-Term Sales Trend**: 
* The forecast shows a gradual drop, indicating shifting consumer preferences
* Strong Seasonal Peaks Remain: Despite long-term decline, certain months still see sharp demand rises, allowing focused marketing efforts.
* Repositioning Needed for Market Retention: Demand declines suggest businesses should innovate, introducing new blends or targeting different demographics.
* Optimizing Inventory & Production: Prevent surplus stock in off-seasons while ensuring high availability during peak demand months.
* Price Sensitivity Consideration: Competitive pricing adjustments may help counteract market shrinkage and attract value-driven consumers.

**Strategic Actions for Sparkling Wine**:
* Seasonal Promotions: Focus sales on high-demand months with exclusive discounts or themed campaigns.
* Brand Reinvention: Adapt product positioning; consider targeting younger demographics or wine-based cocktails to re-engage demand.
* Retail Expansion: Strengthen presence in supermarkets, e-commerce, and lifestyle-focused outlets rather than relying solely on traditional wine sales.
* Market Research & Innovation: Study consumer trends, and test limited editions or subscription-based models for engagement.